# ED tests
Part 1: correctness — ModelSpec against reference QuSpin Hamiltonians.
Part 2: timing — full spectrum on 2D lattices up to 4×4.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))  # add repo root to path

import numpy as np
from quspin.operators import hamiltonian
from quspin.basis import spin_basis_general
import cooling
from cooling.ed import ModelSpec

def make_device(lattice):
    return cooling.CoolingDevice.from_lattice(lattice, Nb=0)

def ref_eigvals(H):
    return np.sort(np.linalg.eigvalsh(H.toarray()).real)

def check(label, E_new, E_ref, atol=1e-5):
    ok = np.allclose(np.sort(E_new), np.sort(E_ref[:len(E_new)]), atol=atol)
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
    if not ok:
        print(f"       new: {np.sort(E_new)[:5]}")
        print(f"       ref: {np.sort(E_ref[:len(E_new)])[:5]}")

header = "  Lx  Ly   PBC   Ns       Hilbert   time (s)"
sep    = "-" * 50

# Part 1: Correctness

## 1. Ising 1D — open chain

In [3]:
L, J, g = 6, 1.0, 0.5
lattice = cooling.ChainLattice1D(L, pbc=False)
model   = cooling.IsingModel(make_device(lattice), {'J': J, 'g': g})
pairs   = lattice.nearest_neighbour_pairs()
basis   = spin_basis_general(L, pauli=True)

# Reference: XX + Z (physical basis)
HJ  = hamiltonian([['xx', [[J, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hg  = hamiltonian([['z',  [[-g, i]  for i in range(L)]]], [], basis=basis, **kw)
E_ref = ref_eigvals(HJ + Hg)

E_new = ModelSpec(model, save=False)
check('Ising 1D OBC, full spectrum', E_new, E_ref)

calculating new energies IsingModel_chain1D_L6_J1.000_g0.500_gx0.000_Spec
  parity symmetry found
  Z2 spin-inversion symmetry found (Hadamard rotation: Z->X)
PASS  Ising 1D OBC, full spectrum


## 2. Ising 1D — periodic chain

In [4]:
L, J, g = 6, 1.0, 0.5
lattice = cooling.ChainLattice1D(L, pbc=True)
model   = cooling.IsingModel(make_device(lattice), {'J': J, 'g': g})
pairs   = lattice.nearest_neighbour_pairs()
basis   = spin_basis_general(L, pauli=True)

HJ  = hamiltonian([['xx', [[J, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hg  = hamiltonian([['z',  [[-g, i]  for i in range(L)]]], [], basis=basis, **kw)
E_ref = ref_eigvals(HJ + Hg)

E_new = ModelSpec(model, save=False)
check('Ising 1D PBC, full spectrum', E_new, E_ref)

calculating new energies IsingModel_chain1D_L6_J1.000_g0.500_gx0.000_Spec
  translation symmetry (x) found
  parity skipped: incompatible with translation symmetry on PBC lattice
  Z2 spin-inversion symmetry found (Hadamard rotation: Z->X)
PASS  Ising 1D PBC, full spectrum


## 3. Ising 1D — with longitudinal field (breaks Z2)

In [5]:
L, J, g, gx = 5, 1.0, 0.5, 0.3
lattice = cooling.ChainLattice1D(L, pbc=False)
model   = cooling.IsingModel(make_device(lattice), {'J': J, 'g': g, 'gx': gx})
pairs   = lattice.nearest_neighbour_pairs()
basis   = spin_basis_general(L, pauli=True)

HJ  = hamiltonian([['xx', [[J,   s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hg  = hamiltonian([['z',  [[-g,  i]   for i in range(L)]]], [], basis=basis, **kw)
Hgx = hamiltonian([['x',  [[-gx, i]   for i in range(L)]]], [], basis=basis, **kw)
E_ref = ref_eigvals(HJ + Hg + Hgx)

E_new = ModelSpec(model, save=False)
check('Ising 1D with X field (Z2 disabled)', E_new, E_ref)

calculating new energies IsingModel_chain1D_L5_J1.000_g0.500_gx0.300_Spec
  parity symmetry found
  Z2 skipped: multiple single-site field types present
PASS  Ising 1D with X field (Z2 disabled)


## 4. ZZ+X Ising (standard QuSpin convention) — Z2 should apply directly

In [6]:
# Define a ZZ+X model by subclassing
from cooling.models import Model

class ZZXIsingModel(Model):
    _ACCEPTED_PARAMS = {'J', 'g'}
    def __init__(self, device, params):
        self.params = params
        self.J = params.get('J', 0.)
        self.g = params.get('g', 0.)
        super().__init__(device)
    @property
    def name(self):
        return f"ZZXIsing_{self.lattice.name}_J{self.J:.3f}_g{self.g:.3f}"
    def build_coupling_lists(self):
        pairs = self._lattice.nearest_neighbour_pairs()
        return {
            'ZZ': [(self.J, s, t) for s, t in pairs],
            'X':  [(-self.g, s) for s in range(self._Ns)]
        }

L, J, g = 6, 1.0, 0.5
lattice = cooling.ChainLattice1D(L, pbc=False)
model   = ZZXIsingModel(make_device(lattice), {'J': J, 'g': g})
pairs   = lattice.nearest_neighbour_pairs()
basis   = spin_basis_general(L, pauli=True)

HJ  = hamiltonian([['zz', [[J,  s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hg  = hamiltonian([['x',  [[-g, i]   for i in range(L)]]], [], basis=basis, **kw)
E_ref = ref_eigvals(HJ + Hg)

E_new = ModelSpec(model, save=False)
check('ZZ+X Ising (Z2 direct, no rotation)', E_new, E_ref)

calculating new energies ZZXIsing_chain1D_L6_J1.000_g0.500_Spec
  parity symmetry found
  Z2 spin-inversion symmetry found
PASS  ZZ+X Ising (Z2 direct, no rotation)


## 5. Heisenberg 1D — open chain

In [7]:
L, Jxx, Jyy, Jzz = 6, 1.0, 1.0, 0.5
lattice = cooling.ChainLattice1D(L, pbc=False)
model   = cooling.HeisenbergModel(make_device(lattice), {'Jxx': Jxx, 'Jyy': Jyy, 'Jzz': Jzz})
pairs   = lattice.nearest_neighbour_pairs()
basis   = spin_basis_general(L, pauli=True)

Hxx = hamiltonian([['xx', [[Jxx, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hyy = hamiltonian([['yy', [[Jyy, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hzz = hamiltonian([['zz', [[Jzz, s, t] for s, t in pairs]]], [], basis=basis, **kw)
E_ref = ref_eigvals(Hxx + Hyy + Hzz)

E_new = ModelSpec(model, save=False)
check('Heisenberg 1D OBC (XXZ)', E_new, E_ref)

calculating new energies HeisenbergModel_chain1D_L6_Jxx1.000_Jyy1.000_Jzz0.500_Spec
  parity symmetry found
  U(1) / Nup symmetry found
  Z2 skipped: incompatible with U(1) Nup sectors
PASS  Heisenberg 1D OBC (XXZ)


## 6. Heisenberg 1D — periodic chain

In [8]:
L, Jxx, Jyy, Jzz = 6, 1.0, 1.0, 1.0
lattice = cooling.ChainLattice1D(L, pbc=True)
model   = cooling.HeisenbergModel(make_device(lattice), {'Jxx': Jxx, 'Jyy': Jyy, 'Jzz': Jzz})
pairs   = lattice.nearest_neighbour_pairs()
basis   = spin_basis_general(L, pauli=True)

Hxx = hamiltonian([['xx', [[Jxx, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hyy = hamiltonian([['yy', [[Jyy, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hzz = hamiltonian([['zz', [[Jzz, s, t] for s, t in pairs]]], [], basis=basis, **kw)
E_ref = ref_eigvals(Hxx + Hyy + Hzz)

E_new = ModelSpec(model, save=False)
check('Heisenberg 1D PBC (isotropic)', E_new, E_ref)

calculating new energies HeisenbergModel_chain1D_L6_Jxx1.000_Jyy1.000_Jzz1.000_Spec
  translation symmetry (x) found
  parity skipped: incompatible with translation symmetry on PBC lattice
  U(1) / Nup symmetry found
  Z2 skipped: incompatible with U(1) Nup sectors
PASS  Heisenberg 1D PBC (isotropic)


## 7. Ising 2D — 3×3 open

In [9]:
Lx, Ly, J, g = 3, 3, 1.0, 0.5
lattice = cooling.SquareLattice2D(Lx, Ly, pbc_x=False, pbc_y=False)
model   = cooling.IsingModel(make_device(lattice), {'J': J, 'g': g})
pairs   = lattice.nearest_neighbour_pairs()
Ns      = Lx * Ly
basis   = spin_basis_general(Ns, pauli=True)

HJ  = hamiltonian([['xx', [[J,  s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hg  = hamiltonian([['z',  [[-g, i]   for i in range(Ns)]]], [], basis=basis, **kw)
E_ref = ref_eigvals(HJ + Hg)

E_new = ModelSpec(model, save=False)
check('Ising 2D 3x3 OBC', E_new, E_ref)

calculating new energies IsingModel_square2D_Lx3Ly3_J1.000_g0.500_gx0.000_Spec
  parity symmetry found
  Z2 spin-inversion symmetry found (Hadamard rotation: Z->X)
PASS  Ising 2D 3x3 OBC


## 8. Heisenberg 2D — 3×3 open

In [10]:
Lx, Ly, Jxx, Jyy, Jzz = 3, 3, 1.0, 1.0, 0.5
lattice = cooling.SquareLattice2D(Lx, Ly, pbc_x=False, pbc_y=False)
model   = cooling.HeisenbergModel(make_device(lattice), {'Jxx': Jxx, 'Jyy': Jyy, 'Jzz': Jzz})
pairs   = lattice.nearest_neighbour_pairs()
Ns      = Lx * Ly
basis   = spin_basis_general(Ns, pauli=True)

Hxx = hamiltonian([['xx', [[Jxx, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hyy = hamiltonian([['yy', [[Jyy, s, t] for s, t in pairs]]], [], basis=basis, **kw)
Hzz = hamiltonian([['zz', [[Jzz, s, t] for s, t in pairs]]], [], basis=basis, **kw)
E_ref = ref_eigvals(Hxx + Hyy + Hzz)

E_new = ModelSpec(model, save=False)
check('Heisenberg 2D 3x3 OBC (XXZ)', E_new, E_ref)

calculating new energies HeisenbergModel_square2D_Lx3Ly3_Jxx1.000_Jyy1.000_Jzz0.500_Spec
  parity symmetry found
  U(1) / Nup symmetry found
  Z2 skipped: incompatible with U(1) Nup sectors
PASS  Heisenberg 2D 3x3 OBC (XXZ)


## 9. Symmetry disable check — same spectrum with symmetries on and off

In [11]:
L, J, g = 6, 1.0, 0.5
lattice = cooling.ChainLattice1D(L, pbc=True)
model   = cooling.IsingModel(make_device(lattice), {'J': J, 'g': g})

print("--- all symmetries on ---")
E_all = ModelSpec(model, save=False)

print("--- all symmetries off ---")
E_none = ModelSpec(model, save=False, translation=False, parity=False, Z2=False, U1=False)

check('Ising 1D PBC: symmetries on vs off', E_all, E_none)

--- all symmetries on ---
calculating new energies IsingModel_chain1D_L6_J1.000_g0.500_gx0.000_Spec
  translation symmetry (x) found
  parity skipped: incompatible with translation symmetry on PBC lattice
  Z2 spin-inversion symmetry found (Hadamard rotation: Z->X)
--- all symmetries off ---
calculating new energies IsingModel_chain1D_L6_J1.000_g0.500_gx0.000_Spec
PASS  Ising 1D PBC: symmetries on vs off


# Part 2: Timing

## Ising 2D  (J=1, g=0.5)

In [2]:
J, g = 1.0, 0.5
cases = [(2,2,False),(3,3,False),(4,3,False),(3,3,True),(4,4,True)]
print(header); print(sep)
for Lx, Ly, pbc in cases:
    lattice = cooling.SquareLattice2D(Lx, Ly, pbc_x=pbc, pbc_y=pbc)
    model = cooling.IsingModel(make_device(lattice), {"J": J, "g": g})
    pbc_str = 'yes' if pbc else 'no'
    print(f'{Lx:>4} {Ly:>3} {pbc_str:>5} {model.Ns:>4} {2**model.Ns:>12,} {run(model):>10.3f}')


  Lx  Ly   PBC   Ns       Hilbert   time (s)
--------------------------------------------------
   2   2    no    4           16      0.328
   3   3    no    9          512      0.021
   4   3    no   12        4,096      1.128
   3   3   yes    9          512      0.077
   4   4   yes   16       65,536    160.589


## Heisenberg 2D  (Jxx=Jyy=Jzz=1)

In [3]:
J = 1.0
cases = [(2,2,False),(3,3,False),(4,3,False),(3,3,True),(4,4,True)]
print(header); print(sep)
for Lx, Ly, pbc in cases:
    lattice = cooling.SquareLattice2D(Lx, Ly, pbc_x=pbc, pbc_y=pbc)
    model = cooling.HeisenbergModel(make_device(lattice), {"Jxx": J, "Jyy": J, "Jzz": J})
    pbc_str = 'yes' if pbc else 'no'
    print(f'{Lx:>4} {Ly:>3} {pbc_str:>5} {model.Ns:>4} {2**model.Ns:>12,} {run(model):>10.3f}')


  Lx  Ly   PBC   Ns       Hilbert   time (s)
--------------------------------------------------
   2   2    no    4           16      0.013
   3   3    no    9          512      0.058
   4   3    no   12        4,096      0.402
   3   3   yes    9          512      0.218
   4   4   yes   16       65,536     23.850


## Symmetries found (verbose)

In [4]:
for label, model_fn, params, Lx, Ly, pbc in [
    ("Ising 4x3 OBC",      cooling.IsingModel,      {"J": 1., "g": .5},             4, 3, False),
    ("Ising 4x4 PBC",      cooling.IsingModel,      {"J": 1., "g": .5},             4, 4, True),
    ("Heisenberg 4x3 OBC", cooling.HeisenbergModel, {"Jxx": 1., "Jyy": 1., "Jzz": 1.}, 4, 3, False),
    ("Heisenberg 4x4 PBC", cooling.HeisenbergModel, {"Jxx": 1., "Jyy": 1., "Jzz": 1.}, 4, 4, True),
]:
    lattice = cooling.SquareLattice2D(Lx, Ly, pbc_x=pbc, pbc_y=pbc)
    model = model_fn(make_device(lattice), params)
    print(f"--- {label} ---")
    ModelSpec(model, save=False, verbal=True)
    print()

--- Ising 4x3 OBC ---
calculating new energies IsingModel_square2D_Lx4Ly3_J1.000_g0.500_gx0.000_Spec
  parity symmetry found
  Z2 spin-inversion symmetry found (Hadamard rotation: Z->X)

--- Ising 4x4 PBC ---
calculating new energies IsingModel_square2D_Lx4Ly4_J1.000_g0.500_gx0.000_Spec
  translation symmetry (x) found
  translation symmetry (y) found
  parity skipped: incompatible with translation symmetry on PBC lattice
  Z2 spin-inversion symmetry found (Hadamard rotation: Z->X)

--- Heisenberg 4x3 OBC ---
calculating new energies HeisenbergModel_square2D_Lx4Ly3_Jxx1.000_Jyy1.000_Jzz1.000_Spec
  parity symmetry found
  U(1) / Nup symmetry found
  Z2 skipped: incompatible with U(1) Nup sectors

--- Heisenberg 4x4 PBC ---
calculating new energies HeisenbergModel_square2D_Lx4Ly4_Jxx1.000_Jyy1.000_Jzz1.000_Spec
  translation symmetry (x) found
  translation symmetry (y) found
  parity skipped: incompatible with translation symmetry on PBC lattice
  U(1) / Nup symmetry found
  Z2 skipped